# End-to-End Credit Scoring MLOps Pipeline

**Главный ноутбук проекта.** Он последовательно запускает подготовку данных,
валидацию, EDA, Feature Engineering, обучение пяти моделей, логирование в MLflow,
DVC pipeline, тесты, REST API и мониторинг PSI.

> Запускайте ячейки сверху вниз из корневой папки репозитория либо откройте ноутбук
> в Jupyter и выполните блок «Подготовка окружения». Обучение нескольких моделей
> и cross-validation может занять несколько минут.

## 0. Подготовка окружения

Ноутбук определяет корень проекта, устанавливает пакет в editable-режиме и проверяет
наличие ключевых директорий. Установка зависимостей выполняется один раз на окружение.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import importlib.util

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError(
        "Не найден корень проекта. Откройте ноутбук из папки credit-scoring-mlops."
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])

print("Устанавливаем проект и dev-зависимости (включая DVC, pytest, Black и Flake8)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], cwd=PROJECT_ROOT, check=True)
print("Зависимости установлены.")

## 1. Импорты и настройки эксперимента

Фиксируем random seed, задаём пути и загружаем функции из модулей проекта.
Вся бизнес-логика находится в `src/credit_scoring`, а ноутбук только оркестрирует её.

In [ ]:
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from credit_scoring.data import download_uci_dataset
from credit_scoring.features import TARGET, engineer_features
from credit_scoring.validation import validate_credit_data
from credit_scoring.prepare import prepare_data
from credit_scoring.train import train_experiments
from credit_scoring.metrics import calculate_metrics
from credit_scoring.monitoring import calculate_psi

RANDOM_STATE = 42
RAW_PATH = PROJECT_ROOT / "data/raw/UCI_Credit_Card.csv"
TRAIN_PATH = PROJECT_ROOT / "data/processed/train.csv"
TEST_PATH = PROJECT_ROOT / "data/processed/test.csv"
MODEL_PATH = PROJECT_ROOT / "models/credit_default_model.joblib"

for folder in ["data/raw", "data/processed", "models", "metrics",
               "reports/figures", "monitoring"]:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
print("Настройки загружены.")

## 2. Загрузка исходного датасета

Используем Default of Credit Card Clients. Если CSV уже находится в `data/raw`,
повторная загрузка не нужна. При недоступности UCI загрузите CSV вручную из Kaggle
и сохраните его под указанным именем.

In [ ]:
if not RAW_PATH.exists():
    print("Исходный файл не найден. Пробуем загрузить датасет из UCI...")
    try:
        download_uci_dataset(RAW_PATH)
    except Exception as exc:
        raise RuntimeError(
            "Автоматическая загрузка не удалась. Скачайте Kaggle CSV и положите "
            f"его в {RAW_PATH}"
        ) from exc

raw = pd.read_csv(RAW_PATH)
raw.columns = [str(c).strip() for c in raw.columns]
if "ID" in raw.columns:
    raw = raw.drop(columns=["ID"])

print(f"Размер датасета: {raw.shape[0]:,} строк × {raw.shape[1]} столбцов")
display(raw.head())

## 3. Первичный EDA

Смотрим типы, пропуски, дубликаты, баланс целевой переменной и базовые распределения.
Это исследовательский блок: окончательная логика подготовки находится в `src/`.

In [ ]:
display(pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_pct": (raw.isna().mean() * 100).round(2),
    "n_unique": raw.nunique()
}).sort_values("missing_pct", ascending=False))

print("Дубликатов строк:", int(raw.duplicated().sum()))
print("\nРаспределение целевой переменной:")
target_share = raw[TARGET].value_counts(dropna=False).sort_index()
display(pd.DataFrame({
    "count": target_share,
    "share_pct": (target_share / len(raw) * 100).round(2)
}))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
target_share.plot(kind="bar", ax=axes[0], title="Целевая переменная")
raw["AGE"].plot(kind="hist", bins=25, ax=axes[1], title="Распределение возраста")
axes[0].set_xlabel("Default (0/1)")
axes[1].set_xlabel("AGE")
plt.tight_layout()
plt.show()

### 3.1. Исследование кредитных и платёжных признаков

Проверяем лимит, возраст, счета, платежи и историю просрочек. Важно: эти графики
помогают сформулировать гипотезы, но не заменяют статистическую проверку.

In [ ]:
numeric_to_plot = ["LIMIT_BAL", "AGE", "BILL_AMT1", "PAY_AMT1",
                   "PAY_0", "PAY_2", "PAY_3"]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, numeric_to_plot):
    raw[col].plot(kind="hist", bins=30, ax=ax, title=col)
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()

default_rate_by_delay = (
    raw.groupby("PAY_0", observed=False)[TARGET].mean().rename("default_rate")
)
display(default_rate_by_delay.to_frame())

## 4. Валидация входных данных (Pandera)

Схема проверяет обязательные поля, отсутствие null в ключевых колонках,
допустимые диапазоны демографических и платёжных статусов, а также бинарную цель.
Отдельно проверяем, что намеренно испорченные данные действительно отклоняются.

In [ ]:
validated = validate_credit_data(raw)
print("Pandera: входной датасет прошёл проверку.")
display(validated.describe().T.head(10))

# Негативная проверка: значение целевой переменной 9 должно вызвать ошибку схемы.
bad_sample = raw.head(20).copy()
bad_sample.loc[bad_sample.index[0], TARGET] = 9
try:
    validate_credit_data(bad_sample)
    raise AssertionError("Ожидалась ошибка валидации, но данные прошли проверку.")
except Exception as exc:
    print("Ожидаемая ошибка на аномальных данных:", type(exc).__name__)

## 5. Feature Engineering и разбиение выборки

Создаём агрегаты истории счетов и платежей, долю платежей относительно суммы счетов,
число месяцев с просрочкой и возрастные интервалы. Затем выполняем стратифицированное
разбиение train/test, сохраняя долю дефолтов.

In [ ]:
engineered = engineer_features(validated)
new_features = [
    "BILL_AMT_MEAN", "BILL_AMT_MAX", "PAY_AMT_MEAN",
    "PAYMENT_TO_BILL_RATIO", "DELAYED_MONTHS", "AGE_BIN"
]
print("Добавленные признаки:")
display(engineered[new_features].head())
display(engineered.groupby(TARGET)[new_features[:-1]].mean(numeric_only=True).T)

train_df, test_df = prepare_data(
    raw_path=str(RAW_PATH),
    output=str(PROJECT_ROOT / "data/processed"),
    test_size=0.2,
    random_state=RANDOM_STATE,
)
print("Train:", train_df.shape, "| Test:", test_df.shape)
print("Доля дефолтов train/test:",
      round(train_df[TARGET].mean(), 4), round(test_df[TARGET].mean(), 4))

## 6. Обучение пяти экспериментов и логирование в MLflow

Для каждого кандидата выполняется cross-validation на train, затем оценка на
отложенном test. В MLflow записываются параметры, метрики, ROC-кривая и модель.
Победитель по ROC-AUC сохраняется в `models/`.

In [ ]:
training_result = train_experiments(
    train_path=str(TRAIN_PATH),
    test_path=str(TEST_PATH),
    random_state=RANDOM_STATE,
    cv=3,
    scoring="roc_auc",
)
display(pd.Series(training_result, name="value").to_frame())
print("Модель сохранена:", MODEL_PATH.exists())

### 6.1. Сравнение экспериментов

Читаем результаты из MLflow Tracking Store. В таблице должны быть минимум пять runs.
Для удобства отдельно показываем ROC-AUC, Precision, Recall, F1 и CV ROC-AUC.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

experiment = mlflow.get_experiment_by_name("credit-default-prediction")
if experiment is None:
    raise RuntimeError("MLflow experiment не найден: проверьте успешность обучения.")

client = MlflowClient()
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.roc_auc DESC"],
)
run_rows = []
for run in runs:
    run_rows.append({
        "run_name": run.info.run_name,
        "run_id": run.info.run_id,
        "model": run.data.params.get("model"),
        "cv_roc_auc_mean": run.data.metrics.get("cv_roc_auc_mean"),
        "roc_auc": run.data.metrics.get("roc_auc"),
        "precision": run.data.metrics.get("precision"),
        "recall": run.data.metrics.get("recall"),
        "f1": run.data.metrics.get("f1"),
    })
experiments_table = pd.DataFrame(run_rows)
display(experiments_table)
assert len(experiments_table) >= 5, "В MLflow должно быть не менее пяти экспериментов."

### 6.2. ROC-кривые

Показываем ROC-кривые, сформированные на отложенной тестовой выборке.
Дополнительные PNG-файлы лежат в `reports/figures/` и также залогированы в MLflow.

In [ ]:
from IPython.display import Image, display as ipy_display

roc_files = sorted((PROJECT_ROOT / "reports/figures").glob("roc_*.png"))
print(f"Сохранено ROC-кривых: {len(roc_files)}")
for image_path in roc_files:
    print(image_path.name)
    ipy_display(Image(filename=str(image_path), width=650))

## 7. DVC: воспроизводимый pipeline

DVC связывает исходные данные, подготовку, обучение, артефакты модели и метрики.
Команды ниже инициализируют DVC (если это ещё не сделано), добавляют исходный CSV
под контроль DVC и запускают `dvc repro`.

> Для `dvc init` необходим Git-репозиторий. Если проект ещё не инициализирован через
> Git, ноутбук создаст локальный репозиторий. Перед публикацией проверьте `.gitignore`.

In [ ]:
def run_command(command, check=True):
    print("$", " ".join(map(str, command)))
    completed = subprocess.run(
        command, cwd=PROJECT_ROOT, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print(completed.stdout[-12000:])
    if check and completed.returncode != 0:
        raise subprocess.CalledProcessError(
            completed.returncode, command, output=completed.stdout
        )
    return completed

if not (PROJECT_ROOT / ".git").exists():
    run_command(["git", "init"])

if not (PROJECT_ROOT / ".dvc").exists():
    run_command(["dvc", "init"])

# dvc add создаёт .dvc-файл и добавляет исключение в .gitignore.
run_command(["dvc", "add", "-f", "data/raw/UCI_Credit_Card.csv"])

In [ ]:
# DVC repro повторно выполнит prepare/train только если входы или параметры изменились.
run_command(["dvc", "repro"])
run_command(["dvc", "dag"], check=False)

dvc_metrics_path = PROJECT_ROOT / "metrics/metrics.json"
if dvc_metrics_path.exists():
    display(pd.Series(json.loads(dvc_metrics_path.read_text()), name="value").to_frame())

## 8. Unit-тесты и локальные CI-проверки

GitHub Actions выполняет эти же проверки на push и pull request. В ноутбуке запускаем
pytest, Black и Flake8 локально. Сам факт локального прохождения не заменяет успешный
workflow run на GitHub — его нужно получить после push.

In [ ]:
run_command([sys.executable, "-m", "pytest", "-q"])
run_command([sys.executable, "-m", "black", "--check", "src", "tests"])
run_command([sys.executable, "-m", "flake8", "src", "tests",
             "--max-line-length=88", "--extend-ignore=E203,W503"])
run_command([sys.executable, "-m", "credit_scoring.validation",
             "--data", "tests/fixtures/valid_credit_sample.csv"])

## 9. FastAPI: запуск и проверка REST API

Запускаем локальный Uvicorn-процесс из ноутбука, ждём готовности `/health`, затем
отправляем тестовую запись в `/predict`. В конце останавливаем сервер.

In [ ]:
import requests
import signal

api_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn",
     "credit_scoring.api.app:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=PROJECT_ROOT,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
API_URL = "http://127.0.0.1:8000"

try:
    health_response = None
    for _ in range(40):
        try:
            health_response = requests.get(f"{API_URL}/health", timeout=2)
            if health_response.ok:
                break
        except requests.RequestException:
            time.sleep(1)
    if health_response is None or not health_response.ok:
        raise RuntimeError("FastAPI не запустился. Проверьте модель и логи Uvicorn.")
    display(health_response.json())

    api_sample = test_df.drop(columns=[TARGET]).iloc[0].to_dict()
    # API-контракт принимает исходные поля; engineered features удаляем.
    api_fields = [
        "LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE",
        "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
        *[f"BILL_AMT{i}" for i in range(1, 7)],
        *[f"PAY_AMT{i}" for i in range(1, 7)],
    ]
    payload = {key: api_sample[key] for key in api_fields}
    predict_response = requests.post(f"{API_URL}/predict", json=payload, timeout=30)
    predict_response.raise_for_status()
    display(predict_response.json())
finally:
    api_process.terminate()
    try:
        api_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        api_process.kill()
        api_process.wait()

### 9.1. Docker (опциональный интеграционный шаг)

Если Docker Engine установлен и запущен, соберите образ и запустите контейнер.
Dockerfile копирует обученную модель, поэтому этот шаг выполняется после обучения.

In [ ]:
# Выполните при установленном и запущенном Docker:
# run_command(["docker", "build", "-t", "credit-scoring-api:local", "."])
# container = run_command([
#     "docker", "run", "-d", "--rm", "-p", "8000:8000",
#     "--name", "credit-scoring-api", "credit-scoring-api:local"
# ])
# try:
#     time.sleep(5)
#     display(requests.get("http://localhost:8000/health", timeout=5).json())
# finally:
#     run_command(["docker", "stop", "credit-scoring-api"], check=False)

## 10. Мониторинг: Population Stability Index (PSI)

Сравниваем распределение предсказанных PD на обучающей выборке с распределением
оценок на новых наблюдениях, отправленных через API. PSI — сигнал сдвига
распределения, но не доказательство ухудшения качества без фактических меток.

In [ ]:
# Для мониторинга поднимаем API на время отправки небольшой тестовой выборки.
api_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn",
     "credit_scoring.api.app:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=PROJECT_ROOT, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
try:
    ready = False
    for _ in range(40):
        try:
            if requests.get(f"{API_URL}/health", timeout=2).ok:
                ready = True
                break
        except requests.RequestException:
            time.sleep(1)
    if not ready:
        raise RuntimeError("API не готов для мониторинга.")

    reference_scores = np.load(PROJECT_ROOT / "monitoring/reference_scores.npy")
    monitoring_sample = test_df.sample(
        n=min(300, len(test_df)), random_state=RANDOM_STATE
    )
    current_scores = []
    for record in monitoring_sample.drop(columns=[TARGET]).to_dict("records"):
        payload = {key: record[key] for key in api_fields}
        response = requests.post(f"{API_URL}/predict", json=payload, timeout=30)
        response.raise_for_status()
        current_scores.append(response.json()["default_probability"])

    psi_value = calculate_psi(reference_scores, current_scores)
    psi_result = {
        "metric": "PSI",
        "population": "predicted_default_probability",
        "psi": round(float(psi_value), 6),
        "reference_size": int(len(reference_scores)),
        "current_size": int(len(current_scores)),
        "interpretation": (
            "stable" if psi_value < 0.1 else
            "moderate_shift" if psi_value < 0.25 else
            "significant_shift"
        ),
    }
    (PROJECT_ROOT / "monitoring/psi_report.json").write_text(
        json.dumps(psi_result, indent=2)
    )
    display(pd.Series(psi_result, name="value").to_frame())
finally:
    api_process.terminate()
    try:
        api_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        api_process.kill()
        api_process.wait()

## 11. Итоговый статус проекта

Собираем основные артефакты, которые стоит показать на защите. Если все предыдущие
ячейки завершились без ошибок, локальный end-to-end сценарий отработал.

In [ ]:
artifacts = {
    "raw_dataset": RAW_PATH.exists(),
    "train_dataset": TRAIN_PATH.exists(),
    "test_dataset": TEST_PATH.exists(),
    "trained_model": MODEL_PATH.exists(),
    "metrics_json": (PROJECT_ROOT / "metrics/metrics.json").exists(),
    "mlflow_roc_artifacts": len(roc_files) >= 5,
    "dvc_config": (PROJECT_ROOT / "dvc.yaml").exists(),
    "dvc_initialized": (PROJECT_ROOT / ".dvc").exists(),
    "psi_report": (PROJECT_ROOT / "monitoring/psi_report.json").exists(),
    "github_actions_workflow": (PROJECT_ROOT / ".github/workflows/ci.yml").exists(),
}
display(pd.DataFrame(
    {"artifact": artifacts.keys(), "created": artifacts.values()}
))
print("\nЛучший эксперимент:")
display(pd.Series(training_result).to_frame("value"))

print("\nСледующие действия вне ноутбука:")
print("1. git add/commit и push в GitHub.")
print("2. Проверить зелёный GitHub Actions workflow.")
print("3. При командной работе настроить DVC remote и выполнить dvc push.")
print("4. Сохранить скриншоты MLflow UI, Swagger, DVC DAG и PSI для защиты.")